In [1]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from pathlib import Path
import sys
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import optuna
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer



from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

sys.path.append(str(Path().resolve().parent))
import src.data_processing.data_processing as dp
import src.data_processing.building_dataset as bd

/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
df_status = pd.read_csv("../data/processed/status.csv")
dp.str_to_int_student_status(df_status)
df_encoding = bd.encoding(df_status, 'onehot', target='student_status')
X = df_encoding.drop(columns=['student_id_hash', 'student_status'])
y = df_encoding['student_status']
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

# model = XGBClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=5,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric="logloss",
#     random_state=42
# )
model = XGBClassifier( random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 200)

best_threshold = 0.5
best_f1 = 0
accuracy = 0

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t
        accuracy = accuracy_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)


print("Лучший порог:", best_threshold)
print("Лучший F1:", best_f1)
print("accuracy: ", accuracy )
print('recall: ', recall)
print('precision: ', precision )
y_final = (y_proba >= best_threshold).astype(int)


Лучший порог: 0.5384422110552763
Лучший F1: 0.8491321762349799
accuracy:  0.9910217702208803
recall:  0.803030303030303
precision:  0.9008498583569405


In [35]:
model.get_params()

{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': None,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': None,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': None,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': 42,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [45]:

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

model = GradientBoostingClassifier(n_estimators = 108, 
                                   learning_rate = 0.03736563332951993, 
                                     max_depth = 8, min_samples_split = 13,
                                       min_samples_leaf= 6,
                                         subsample= 0.6816427025399264,
                                         max_features = None)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 200)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Лучший порог:", best_threshold)
print("Лучший F1:", best_f1)
y_final = (y_proba >= best_threshold).astype(int)




Лучший порог: 0.6741206030150754
Лучший F1: 0.8449931412894376


In [ ]:
df_status = pd.read_csv("../data/processed/status.csv")
dp.str_to_int_student_status(df_status)
df_encoding = bd.encoding(df_status, 'onehot', target='student_status')
X = df_encoding.drop(columns=['student_id_hash', 'student_status'])
y = df_encoding['student_status']
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 200)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Лучший порог:", best_threshold)
print("Лучший F1:", best_f1)
y_final = (y_proba >= best_threshold).astype(int)
